In [19]:
import boto3
from botocore.exceptions import ClientError
import json
import mysql.connector
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

def getSecret():

    secret_name = "ufcDBcred"
    region_name = "us-west-1"

    #create client
    session = boto3.session.Session()
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        raise e

    #format as dict before returning
    secret = get_secret_value_response['SecretString']
    return json.loads(secret)

dbCredentials = getSecret()
cnx = mysql.connector.connect(
    user=dbCredentials['username'],
    password=dbCredentials['password'],
    host=dbCredentials['host'],
    database=dbCredentials['dbInstanceIdentifier']
)
cursor = cnx.cursor()

print('We connected with secrets manager!')

We connected with secrets manager!


In [20]:
query = """
SELECT *
from fighterHyperlinks
"""
cursor.execute(query)
result = cursor.fetchall()


column_names = [column[0] for column in cursor.description]
fighters = pd.DataFrame(result, columns=column_names)

In [21]:
query = """
SELECT *
from fightStats
"""
cursor.execute(query)
result = cursor.fetchall()


column_names = [column[0] for column in cursor.description]
fights = pd.DataFrame(result, columns=column_names)

In [22]:
query = """
SELECT *
from eventHyperlinks
"""
cursor.execute(query)
result = cursor.fetchall()


column_names = [column[0] for column in cursor.description]
events = pd.DataFrame(result, columns=column_names)

### These are some metrics we can look into calculating
- Win/Loss Ratio: The ratio of the number of wins to the number of losses for each fighter. This could be a good indicator of a fighter’s overall performance.
- Average Fight Time: The average duration of all fights for each fighter. This could indicate a fighter’s stamina or their ability to end fights quickly.
- Age: The age of the fighter at the time of the fight. Age could impact a fighter’s performance.
- Experience: The total number of fights each fighter has had. More experienced fighters may have an advantage.
- Knockout Rate: The number of fights won by knockout divided by the total number of fights. This could indicate a fighter’s striking power.
- Submission Rate: The number of fights won by submission divided by the total number of fights. This could indicate a fighter’s grappling skill.
- Significant Strikes Difference: The difference between significant strikes landed per minute and strikes absorbed per minute. A positive value could indicate a fighter’s striking effectiveness.
- Takedown Difference: The difference between takedown average and takedowns absorbed. A positive value could indicate a fighter’s wrestling dominance.
- Weight Class: The weight class of the fight. Some fighters perform better in certain weight classes.
- Stance: The fighting stance of the fighter (Orthodox, Southpaw, etc.). This could impact the fight dynamics.
- winstreak

In [23]:
newFighterStats = fighters[['fighterID', 'firstName', 'lastName']]
newFighterStats['fullName'] = newFighterStats['firstName'] + " " + newFighterStats['lastName']
newFighterStats.drop(columns=['firstName', 'lastName'], inplace=True)
newFighterStats[newFighterStats['fullName'] == 'Israel Adesanya']

,fighterID,fullName
19,20,Israel Adesanya


In [41]:
#given some fighterID we can query the name from the newFighterStats view and pass those in as parameters to get the win percentage
currentID = 20
fighterName = newFighterStats.loc[newFighterStats['fighterID'] == currentID, 'fullName'].values[0]
query = f"""
SELECT AVG(winCount) as `win percentage`
FROM (
    SELECT 
    CASE
        WHEN winner = '{fighterName}' THEN 1
        ELSE 0
    end as winCount
    from fightStats
    where fighter_A_ID = {currentID} or fighter_B_ID = {currentID}
) as innerQuery
"""
cursor.execute(query)
result = cursor.fetchall()


column_names = [column[0] for column in cursor.description]
fights = pd.DataFrame(result, columns=column_names)
fights['win percentage'][0]

Decimal('0.8125')

In [47]:
def winPercentageCalculator(row):
    fighterID = row['fighterID']
    fighterName = row['fullName']
    query = f"""
    SELECT AVG(winCount) as `win percentage`
    FROM (
        SELECT 
        CASE
            WHEN winner = "{fighterName}" THEN 1
            ELSE 0
        end as winCount
        from fightStats
        where fighter_A_ID = {fighterID} or fighter_B_ID = {fighterID}
    ) as innerQuery
    """
    cursor.execute(query)
    result = cursor.fetchall()

    column_names = [column[0] for column in cursor.description]
    fights = pd.DataFrame(result, columns=column_names)
    win_percentage = fights['win percentage'][0]
    if win_percentage is None:
        return None
    return win_percentage

newFighterStats['win_percentage'] = newFighterStats.apply(winPercentageCalculator, axis=1)


In [48]:
newFighterStats

,fighterID,fullName,win_percentage
0,1,Tom Aaron,None
1,2,Danny Abbadi,0.0000
2,3,Nariman Abbasov,None
3,4,David Abbott,0.0000
4,5,Hamdy Abdelwahab,0.0000
...,...,...,...
4157,4158,Dave Zitanick,None
4158,4159,Alex Zuniga,None
4159,4160,George Zuniga,None
4160,4161,Allan Zuniga,0.0000


In [49]:
newFighterStats[newFighterStats['win_percentage'] >= 0]

,fighterID,fullName,win_percentage
1,2,Danny Abbadi,0.0000
3,4,David Abbott,0.0000
4,5,Hamdy Abdelwahab,0.0000
6,7,Shamil Abdurakhimov,0.4545
8,9,Daichi Abe,0.3333
...,...,...,...
4150,4151,Zhalgas Zhumagulov,0.1429
4151,4152,Fares Ziam,0.7143
4153,4154,James Zikic,0.0000
4155,4156,Cat Zingano,0.4286
